In [34]:
import torch
import pandas as pd
import numpy as np
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(torch.cuda.get_device_name())

NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [35]:
df = pd.read_parquet("/home/cry_more/ongoing/LogRecall/artifacts/stage1_triplet_data.parquet")

In [41]:
from torch.utils.data import DataLoader,Dataset
import random
from sklearn.model_selection import train_test_split

class TripletDataset(Dataset):
    def __init__(self,X,y):
        self.text = X
        self.hash = y

        self.hash_to_ind = self.hash.groupby(self.hash).groups
        self.all_hash = list(self.hash.unique())

    def __len__(self):
        return len(self.text)
    
    def __getitem__(self, index):
        anchor_text = self.text.iloc[index]
        anchor_hash = self.hash.iloc[index]

        pos_text = anchor_text

        neg_hash = random.choice(self.all_hash)
        while neg_hash==anchor_hash:
            neg_hash = random.choice(self.all_hash)
        
        neg_idx = random.choice(self.hash_to_ind[neg_hash])
        neg_text = self.text.iloc[neg_idx]

        return anchor_text,pos_text,neg_text

X=df['masked_log']
y=df['hash_id']
X_train,X_val,y_train,y_val =  train_test_split(X,y,test_size=.20,random_state=33)
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)

train_dataset = TripletDataset(X_train,y_train)
val_dataset = TripletDataset(X_val,y_val)

In [ ]:
train_dataloader = DataLoader(train_dataset,num_workers=4,batch_size=32,collate_fn=custom_collate)
val_dataset = DataLoader(val_dataset,num_workers=4,batch_size=32,collate_fn=custom_collate)